# Notebook 01: Victoria Electricity Demand — Data Preprocessing and Exploration

**Project**: MSTL Implementation and Extensions  
**Institution**: Indian Institute of Technology Guwahati (IIT Guwahati)  
**School**: Mehta Family School of Data Science and Artificial Intelligence  
**Program**: B.Tech in Data Science and Artificial Intelligence  
**Authors**:
- Rup Narayan Jha (Roll No: 240150032)
- Jaypal Malviya (Roll No: 240150013)
- Anuj Kumar (Roll No: 240150003)
- Ravi Kiran (Roll No: 240150029)

---

## 1. Introduction and Objectives
In this notebook, we load, inspect, preprocess, and explore the **Victoria Electricity Demand** (`vic_elec`) dataset used in the benchmark paper:
> *Bandara, K., Hyndman, R. J., & Bergmeir, C. (2021). MSTL: A Seasonal-Trend Decomposition Algorithm for Time Series with Multiple Seasonal Patterns.*

We perform:
1. Loading the raw half-hourly electricity demand dataset from the Australian Energy Market Operator (AEMO).
2. Converting Excel serial dates and period numbers (1–48) to UTC datetime timestamps.
3. Summing Operational and Industrial demand components into total electricity demand.
4. Aggregating half-hourly data into hourly data via summing.
5. Handling missing values via linear interpolation.
6. Extracting exactly 3,601 observations (2014-01-01 00:00:00 to 2014-05-31 00:00:00) corresponding to the 149-day experimental period with inclusive boundaries.
7. Visualizing full series, 2-week subseries, and daily/weekly demand distributions.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('../src'))

from data_processing import load_raw_dataset, preprocess_to_hourly, extract_experiment_period

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
sns.set_theme(style='whitegrid')

## 2. Load Raw Half-Hourly Dataset
The raw dataset contains 230,784 half-hourly records spanning 2002 to 2015.

In [ ]:
raw_csv_path = '../data/vic_elec_demand.csv'
df_raw = load_raw_dataset(local_path=raw_csv_path)
print(f"Raw shape: {df_raw.shape}")
df_raw.head()

## 3. Preprocess and Aggregate to Hourly
We parse dates, calculate total demand, aggregate half-hours to hourly sums, and interpolate missing values.

In [ ]:
hourly = preprocess_to_hourly(df_raw)
print(f"Total hourly records: {len(hourly)}")
print(f"Start: {hourly.index[0]} | End: {hourly.index[-1]}")
hourly.head()

## 4. Extract Experiment Window (Exact 3,601 Observations)
The research paper evaluates a 149-day window in 2014 from January 1 to May 31. With inclusive endpoints (00:00 to 00:00), this yields exactly 3,601 hourly observations.

In [ ]:
series = extract_experiment_period(hourly, n_obs=3601)
print(f"Extracted length: {len(series)}")
print(f"Start timestamp: {series.index[0]}")
print(f"End timestamp:   {series.index[-1]}")
assert len(series) == 3601, f"Expected 3601 observations, got {len(series)}"
series.describe()

## 5. Time Series Visualizations
Let us plot the full 3,601-observation series, as well as a 2-week zoom to clearly see the overlapping daily (24h) and weekly (168h) patterns.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(series.index, series.values, color='steelblue', lw=0.8)
ax.set_title("Victoria Electricity Demand — 3,601 Hourly Observations (Jan-May 2014)", fontweight='bold')
ax.set_ylabel("Demand (MWh)")
ax.set_xlabel("Time")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
two_weeks = series.iloc[:336]
ax.plot(two_weeks.index, two_weeks.values, color='darkorange', lw=1.2)
ax.set_title("Victoria Electricity Demand — First Two Weeks (336 hours)", fontweight='bold')
ax.set_ylabel("Demand (MWh)")
ax.set_xlabel("Time")
plt.tight_layout()
plt.show()

## 6. Summary of Findings
- The series displays pronounced dual seasonality:
  - **Daily cycle ($m_1 = 24$)**: Distinct morning and evening demand peaks.
  - **Weekly cycle ($m_2 = 168$)**: Significant demand drop during weekends.
- Mean demand is 11,278 MWh with standard deviation of 1,997 MWh.
- Data is clean, strictly positive, with 0 NaNs after preprocessing.